# Suggesting the Expense Type for Department for Transport spend lines

**Patrick Gilliland**

**Question.** Can a model trained on the Department for Transport's past ledger lines suggest the right Expense Type for a new line over £25,000, and what share of lines could it code confidently enough to leave the rest to a person?

**Task 1: the problem and the dataset.** Every payment line a finance team posts carries an account code, which in this data is the `Expense Type`. The problem is to suggest that code for a new line from the other fields on the line, such as the supplier and the item text. In accounting terms this is general ledger account classification: assigning an account code to each posting line. The task is supervised learning (classification), and it is multi-class, because `Expense Type` holds many labels. The dataset is [Spend over £25,000 in the Department for Transport and its arms-length bodies](https://www.data.gov.uk/dataset/804be0c6-ae01-4899-bbb9-2317dff1a4f9/financial-transactions-data-dft), which the Department for Transport publishes on data.gov.uk as one CSV file a month. I use the 48 months from April 2022 to March 2026. `build_dataset.py` stacks the original monthly files in `dft_originals/` into `dft_spend_over_25k_2022-04_to_2026-03.csv`, beside this notebook, without changing any cell. Contains public sector information licensed under the Open Government Licence v3.0.

**Report.** The Machine Learning Analysis Report is one PDF under the two names the brief uses: `module_summary.pdf`, and an identical copy named `Machine_Learning_Analysis_Report.pdf`.

## Data preparation

### Setup

In [1]:
import hashlib
import os
import platform
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

# I show every column and whole cell text, and figures with two decimal places
# and thousands separators, so the wide tables below read without scientific
# notation.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Every step below that draws at random takes this seed, so a rerun gives the
# same numbers.
SEED = 42

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"Matplotlib {matplotlib.__version__}")
print(f"Random seed: {SEED}")
print(f"CPU cores: {os.cpu_count()}")

Python 3.12.3
NumPy 2.5.3
pandas 3.0.6
scikit-learn 1.9.1
Matplotlib 3.11.2
Random seed: 42
CPU cores: 24


**Hardware.** I run everything on the CPU, with no GPU. The models are scikit-learn's, and the whole file fits in memory, so I load it in one go. Where a scikit-learn step can use every core, it gets `n_jobs=-1`.

### Task 2: Load and inspect the dataset

In [2]:
DATA = Path("dft_spend_over_25k_2022-04_to_2026-03.csv")
EXPECTED_SHA256 = (
    "3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4"
)

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} is not beside this notebook. Build it from the 48 original "
        "monthly files in dft_originals/ with: python build_dataset.py"
    )
digest = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"File: {DATA}, {DATA.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")
# Every figure below comes from this exact file, so I stop rather than
# analyse a different one.
if digest != EXPECTED_SHA256:
    raise ValueError(
        f"{DATA} is not the file build_dataset.py writes from the 48 original "
        "files. Rebuild it with: python build_dataset.py"
    )
print(
    "It matches the file build_dataset.py writes from the 48 original files."
)

# I read every cell as the text DfT published. The amounts carry a pound sign
# and commas, and by default pandas would turn text such as "#N/A" into a
# blank, so nothing is converted on load.
df = pd.read_csv(DATA, dtype=str, keep_default_na=False)
print(f"Loaded: {len(df):,} rows, {df.shape[1]} columns")

File: dft_spend_over_25k_2022-04_to_2026-03.csv, 25,396,696 bytes
SHA-256: 3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4
It matches the file build_dataset.py writes from the 48 original files.
Loaded: 126,346 rows, 11 columns


In [3]:
df.head()

,Department Family,Entity,Date,Expense Type,Expense Area,Supplier,Transaction No,Item Text,£,Postal Code,source_file
0,Department for Transport,Department for Transport,05/04/2022,Grt Aid to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£418,408,000.00",NW1 2DN,dft_2022-04.csv
1,Department for Transport,Department for Transport,29/04/2022,Grt Aid to NDPBs,"DG Roads, Places and Environment",National Highways Limited,2000003150,Grant in Aid NH for May 2022,"£298,000,000.00",B1 1RN,dft_2022-04.csv
2,Department for Transport,Department for Transport,29/04/2022,CurrGrtOth to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000003127,NR Grant - DUE 03/05/2022 - SR 9000953144,"£151,086,319.00",NW1 2DN,dft_2022-04.csv
3,Department for Transport,High Speed Two (HS2) Limited,07/04/2022,AUC - Phase 1,000000 AUC - Phase 1,BALFOUR BEATTY VINCI JV - HS2 (N2),#,MWCC Stage One TP,"£135,660,641.62",#,dft_2022-04.csv
4,Department for Transport,Department for Transport,05/04/2022,Cap Grt NDPBs & CG,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£117,640,000.00",NW1 2DN,dft_2022-04.csv


## Modeling

## Evaluation

## Outputs

## Summary